> **Executed output.** Exported from Databricks job `northbay-end-to-end` run 59476289597609, task `governance` (task run 48639267579217), started 2026-11-08 09:41 UTC, duration 66s, result **SUCCESS**, serverless compute.


# 03 · Govern the data product in Unity Catalog

* **Discoverability**: table and column comments, tags for domain / data product / sensitivity.
* **Least-privilege access**: the business app's service principal gets read-only access to the gold layer only
  (Genie queries run with it); bronze/silver stay private to the data engineering team.
* **Lineage**: captured automatically by Unity Catalog from raw files through the pipeline, the stress test and
  the Lakebase synced tables; queried below from the lineage system table.

In [ ]:
dbutils.widgets.text("catalog", "classic_stable_z57t9z_catalog")
dbutils.widgets.text("schema", "febar_supply_chain")
dbutils.widgets.text("app_name", "northbay-resilience")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
app_name = dbutils.widgets.get("app_name")
fq = f"{catalog}.{schema}"

In [ ]:
column_comments = {
    "gold_network_nodes": {
        "node_id": "Network node id: FGxx finished-goods plant, S2xx Tier-2 supplier, R3xx Tier-3 supplier",
        "tier": "1 = finished-goods plant, 2 = co-packer / converter, 3 = raw material supplier",
        "capacity_per_week": "Maximum output per week (cases for plants, case-equivalents for suppliers)",
        "on_hand_units": "Latest on-hand inventory from the WMS snapshot",
        "demand_per_week": "Average forecast demand over the next 4 weeks (plants only)",
        "margin_usd_per_unit": "Gross margin per case in USD (plants only)",
        "expected_recovery_weeks": "Weeks to restore the supplier after its typical disruption (time to recover, TTR)",
    },
    "gold_stress_test_results": {
        "recovery_weeks": "Time to recover (TTR): weeks the disrupted nodes produce nothing",
        "time_to_survive_weeks": "Time to survive (TTS): weeks demand is fully met without the nodes (52 = whole year)",
        "exposure_gap_weeks": "max(TTR - TTS, 0): weeks the business is exposed",
        "lost_margin_usd": "Unavoidable gross margin lost during recovery after optimal re-planning",
        "lost_revenue_usd": "Revenue of the lost cases at list price",
        "risk_rank": "Rank by gross margin lost within the scenario type (1 = most costly)",
    },
    "gold_sourcing_exposure": {
        "n_suppliers": "Approved suppliers for this (node, material); 1 = single point of failure",
    },
}
def alter(table, clause):
    """Pipeline-managed gold tables are materialized views; stress-test outputs are Delta tables."""
    try:
        spark.sql(f"ALTER TABLE {fq}.{table} {clause}")
    except Exception:
        spark.sql(f"ALTER MATERIALIZED VIEW {fq}.{table} {clause}")


for table, cols in column_comments.items():
    for col, text in cols.items():
        alter(table, f"ALTER COLUMN {col} COMMENT '{text}'")

# Keys are prefixed so they do not collide with workspace-wide governed tag policies (e.g. a governed `domain` key)
tags = {"nb_domain": "supply_chain", "nb_data_product": "northbay_network_resilience", "nb_owner": "supply_chain_analytics"}
gold = [r.tableName for r in spark.sql(f"SHOW TABLES IN {fq}").collect() if r.tableName.startswith("gold_")]
for key, value in tags.items():
    try:
        spark.sql(f"ALTER SCHEMA {fq} SET TAGS ('{key}' = '{value}')")
        for t in gold:
            alter(t, f"SET TAGS ('{key}' = '{value}')")
        print(f"tag {key}={value} on schema + {len(gold)} gold tables")
    except Exception as e:  # governed tag policies may restrict keys/values in this workspace
        print(f"skipped tag {key}: {str(e).splitlines()[0][:160]}")

tag nb_domain=supply_chain on schema + 6 gold tables
tag nb_data_product=northbay_network_resilience on schema + 6 gold tables
tag nb_owner=supply_chain_analytics on schema + 6 gold tables


## Least-privilege grants for the business app

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
app_sp = w.apps.get(app_name).service_principal_client_id
spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO `{app_sp}`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {fq} TO `{app_sp}`")
for t in gold:
    spark.sql(f"GRANT SELECT ON TABLE {fq}.{t} TO `{app_sp}`")
print(f"app service principal {app_sp}: USE CATALOG, USE SCHEMA, SELECT on {len(gold)} gold tables")

app service principal 9b7a7976-0615-46cc-b680-3df7ca1a2c0c: USE CATALOG, USE SCHEMA, SELECT on 6 gold tables


In [ ]:
print(spark.sql(f"""
  SELECT t.table_name, t.table_type, t.comment, concat_ws(', ', collect_list(concat(g.tag_name, '=', g.tag_value))) AS tags
  FROM {catalog}.information_schema.tables t
  LEFT JOIN {catalog}.information_schema.table_tags g
    ON g.schema_name = t.table_schema AND g.table_name = t.table_name
  WHERE t.table_schema = '{schema}' AND t.table_name NOT LIKE '\\_\\_%' AND t.table_name NOT LIKE 'event_log%'
  GROUP BY ALL ORDER BY t.table_name""").toPandas().to_string(index=False, max_colwidth=70))

               table_name        table_type                                                                comment                                                                   tags
   bronze_demand_forecast   STREAMING_TABLE                                        Raw APS 13-week demand forecast                                                                       
         bronze_inventory   STREAMING_TABLE                Raw WMS weekly on-hand inventory snapshots (JSON lines)                                                                       
         bronze_materials   STREAMING_TABLE                                                Raw SRM material master                                                                       
          bronze_products   STREAMING_TABLE                                Raw ERP product & plant master extracts                                                                       
         bronze_scenarios   STREAMING_TABLE   Raw named disruption sce

In [ ]:
print(spark.sql(f"""
  SELECT privilege_type, table_name, grantee
  FROM {catalog}.information_schema.table_privileges
  WHERE table_schema = '{schema}' AND grantee = '{app_sp}' ORDER BY table_name""").toPandas().to_string(index=False))

privilege_type                table_name                              grantee
        SELECT gold_disruption_scenarios 9b7a7976-0615-46cc-b680-3df7ca1a2c0c
        SELECT        gold_network_nodes 9b7a7976-0615-46cc-b680-3df7ca1a2c0c
        SELECT       gold_product_impact 9b7a7976-0615-46cc-b680-3df7ca1a2c0c
        SELECT    gold_sourcing_exposure 9b7a7976-0615-46cc-b680-3df7ca1a2c0c
        SELECT  gold_stress_test_results 9b7a7976-0615-46cc-b680-3df7ca1a2c0c
        SELECT         gold_supply_links 9b7a7976-0615-46cc-b680-3df7ca1a2c0c


## Lineage (from `system.access.table_lineage`; can lag the first run by a few minutes)

In [ ]:
print(spark.sql(f"""
  SELECT DISTINCT coalesce(source_table_full_name, source_path) AS source,
         target_table_full_name AS target, entity_type
  FROM system.access.table_lineage
  WHERE (target_table_full_name LIKE '{fq}.%' OR target_table_full_name LIKE '{catalog}.northbay_serving.%')
    AND target_table_full_name NOT LIKE '%event_log%' AND target_table_full_name NOT LIKE '%__materialization%'
    AND event_date >= current_date() - 7
  ORDER BY target, source""").toPandas().to_string(index=False, max_colwidth=90))

                                                                   source                                                                     target entity_type
                                                                     None    classic_stable_z57t9z_catalog.febar_supply_chain.bronze_demand_forecast    PIPELINE
                                                                     None          classic_stable_z57t9z_catalog.febar_supply_chain.bronze_inventory    PIPELINE
                                                                     None          classic_stable_z57t9z_catalog.febar_supply_chain.bronze_materials    PIPELINE
                                                                     None           classic_stable_z57t9z_catalog.febar_supply_chain.bronze_products    PIPELINE
                                                                     None          classic_stable_z57t9z_catalog.febar_supply_chain.bronze_scenarios    PIPELINE
                                  